# 🫀 퀘스트 46 · Q4-E — **동작점, 그리고 축이 맞는가**

| | **MedKOS / `notebooks/quest46_q4e_operating_point.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층② 점수 눈금 |
| 부모 런 | `quest46_q4d_calibrator_shift_equivariance`(`20260805T0300`) |
| 성격 | **동작점 최초 측정 + 축 결정** — 지표를 읽는 법과 다음 축을 함께 정한다 |

## 왜 이 런인가 — 두 가지가 걸려 있다

### (실무) 위양성·위음성을 **한 번도 안 쟀다**

Q3~Q4-D 는 전부 **문턱 없는 지표**(PR-AUC · AUROC)였다. 민감도도 PPV도 환자당
오경보율도 측정한 적이 없다. 「실제 모델에 쓸 수 있나」에 답하려면 **동작점**이 필요하다.

### (전략) **매크로 0.58 이 무작위에 가까운가**

아니다 — **PR-AUC 의 무작위 기저선은 0.5 가 아니라 그 레코드의 유병률**이다.

```
유병률 0.007 → 무작위 AP 0.007   (관측 0.58 이면 83배)
유병률 0.10  → 무작위 AP 0.10    (5.8배)
유병률 0.576 → 무작위 AP 0.576   (1.0배)  ← 레코드 48. 여기는 거의 무작위가 맞다
```

유병률을 0.0070~0.5764 로 섞은 매크로 **하나로는 읽을 수 없다**. 그래서 이 런은
**레코드마다 유병률·PR-AUC·AUROC·lift 를 같이 낸다**(H1).

⚠️ 그리고 **퀘스트 안에서 지표가 갈렸다** — Q2/Q7-B′ 의 「환자 매크로 **0.8842**」는
`roc_auc_score` 이고 Q4 라인의 「매크로 **0.5796**」은 `average_precision_score` 다.
**서로 비교할 수 없는 수**인데 같은 이름으로 불려 왔다. H1 이 같은 코호트에서 **둘 다**
내서 다리를 놓는다.

## ★★★ 축 결정 — 층②는 매크로를 **원리적으로** 못 움직인다

Q4-D 의 G0 가 이미 보였다: 레코드별 상수 로짓 시프트는 매크로를 **정확히** 불변으로
둔다(두 보정기 통틀어 max|Δ| = 0.0e+00). 그런데 **층②(사전확률 정렬·부담 주입)가
바로 그 부류**다. 즉

> **층②로 할 수 있는 일에 매크로는 애초에 없다.** 층②가 움직일 수 있는 건
> **레코드 간 비교 가능성**뿐이고, 실제로 그건 움직였다(`TE − A_em` 교차 **+0.2002**).

H2 가 이걸 **임의의 무작위 시프트**로 다시 확인하고(구성), H3 이 **층②에 남은 천장**을
잰다 — 교차레코드를 직접 최대화하는 오라클 시프트. `TE` 가 그 천장에 붙어 있으면
**층②는 끝난 것**이고, 다음은 층①(표현) 아니면 층④(코호트)다.

## 관문 (사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **H0** | 코호트 · **Platt 기울기 `a > 0`** | 구성. 깨지면 **중단**(Q4-D 의 미검사 결함) |
| **H1 ★★★ 눈금** | 레코드별 유병률 · PR-AUC · AUROC · lift | 관문 아님. **읽는 법을 세운다** |
| **H2 ★★★ 축** | 임의 레코드별 시프트에서 매크로 불변 | 허용 1e-12. 깨지면 **중단** |
| **H3 ★★ 남은 천장** | 교차레코드를 직접 최대화하는 오라클 시프트 | 관문 아님. **상한**이지 방법이 아니다 |
| **H4 ★★★ 동작점 A** | **환자별 예산**(각 레코드 상위 q%) — 민감도·PPV | 관문 아님. 매크로가 정하는 것 |
| **H5 ★★★ 동작점 B** | **전역 단일 문턱**(DEV 에서만 결정) — 민감도·PPV·**산포** | `TE − A_em` 이 측정된 영점 상단 초과 |
| **H6** | 결론 검산표 | R38 ⑦ · R39 ⑤ |

### 판정표

- **H3 에서 `TE` ≈ 천장** → **층②는 끝났다.** 다음 축은 층①(표현) 또는 층④(코호트)
- **H3 에서 여유가 크다** → 층②에 아직 할 일이 있다(시프트를 유병률이 아닌 것으로)
- **H5 ✅** → 단일 문턱을 쓸 때 `TE` 가 환자 간 동작점을 **실제로** 고르게 만든다
- **H5 ❌/미결** → 교차레코드 이득이 동작점으로 **전환되지 않는다**(R40 ① 의 사례)

### 사전등록 상수 (R39 ① · R34 ②)

- 예산 `FLAG_Q = (0.05, 0.10)` — **미리 고정**. TEST 에서 쓸어보지 않는다
- 전역 문턱은 **그 fold 의 DEV 에서만** 결정한다. held-out 은 안 본다
- 주 지표는 `FLAG_Q = 0.05` · 보정기 `platt` · 대비 `TE − A_em`

⚠️ **새 데이터 0** — `svdb_data5.npz` 만.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(thr)):
        return "⚠️ 미결"
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def boot_sd_diff(a, b, seed, nb=3000, q=2.5):
    """★ **산포의 차** — 단일 문턱의 일관성이 여기서 보인다(SD(b) − SD(a))."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 4:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [float(b[j].std(ddof=1) - a[j].std(ddof=1))
         for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float(b.std(ddof=1) - a.std(ddof=1)), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def need_super(n, half, eff, p80=False):
    """`eff` 는 **관문 문턱과의 거리**다(영점 평균이 아니라) — Q4-B 오류의 정정."""
    if not np.isfinite(half) or not np.isfinite(eff) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

def derangement(n, rng):
    for _ in range(1000):
        p = rng.permutation(n)
        if not np.any(p == np.arange(n)):
            return p
    return np.roll(np.arange(n), 1)

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260805, 1
RHY_K = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25

# ── ★ 사전등록 상수 (SMOKE 가 절대 안 건드린다 · R39 ① · R34 ②)
TOL_IDENT = 1e-12
DEV_EVERY = 4
FLAG_Q = (0.05, 0.10)      # ★★ 경보 예산을 **미리 고정**. TEST 에서 쓸어보지 않는다
MAX_NEG_SLOPE = 0.10       # ★ Platt 기울기가 음수인 fold 의 허용 비율(사전 고정)
MAIN_Q = 0.05              # 주 지표로 읽을 예산

# ── 비용 손잡이
NB_BOOT = 400 if SMOKE else 2000
N_PERM  = 2   if SMOKE else 10
N_GRID  = 9   if SMOKE else 21     # H3 오라클 시프트 격자

CALS = ("iso", "platt")
PRIMARY_CAL = "platt"      # Q4-D 결론 — 순위를 보존하고 ECE 도 같거나 낫다
BSPEC = {"TE": ("true", "em"), "TT": ("true", "true")}
ARMS = ("raw", "A_em") + tuple(BSPEC)
MAIN = ("A_em", "TE")      # 배포 가능판끼리
READ_ORDER = ("H0", "H1", "H2", "H3", "H4", "H5", "H6")

SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(   # Q4-D(`20260805T0300`) 실측 — 같은 코호트 · 재현 앵커
    n_ok=56, dom_rec=48,
    q4d_macro=dict(iso=dict(raw=0.5117, A_em=0.5117, TE=0.5663, TT=0.5698),
                   platt=dict(raw=0.5796, A_em=0.5796, TE=0.5772, TT=0.5772)),
    q4d_xrec=dict(iso=dict(raw=0.8829, A_em=0.7501, TE=0.8980, TT=0.8845),
                  platt=dict(raw=0.8940, A_em=0.7182, TE=0.9184, TT=0.8994)),
    q4d_iso_damage=0.0679, q4d_platt_spread=0.0,
    q4d_TE_minus_Aem_xrec_platt=0.2002,
    q7_macro_auroc=0.8842,     # ⚠️ Q2/Q7-B′ 는 **AUROC** 다(Q4 라인은 PR-AUC)
    q7_note="roc_auc_score · 72개체 · `ailab-2026-0053`")

RULE_CHECK = {
    "R11 매크로":       "매크로는 **레코드별 기저선(유병률)과 함께** 읽는다 — 단독은 못 읽는다",
    "R16 fallback 없음": "자산 없으면 **중단**",
    "R22 누출 없음":     "★★★ **문턱을 그 fold 의 DEV 에서만** 결정한다(held-out 은 안 본다)",
    "R26 / R38 ②":      "대비의 **영점**을 rep×레코드로 측정. 못 쟀으면 **안 읽는다**",
    "R29 ② 분기 금지":   "H0 · H2 가 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. **미결 ≠ 등가**",
    "R34 ② 문턱 금지":  "★★★ 예산 `FLAG_Q` 를 **사전 고정**. TEST 에서 쓸어보지 않는다",
    "R35 ① 자 먼저":    "★★ **H1 이 눈금이다** — 유병률 없이는 PR-AUC 를 못 읽는다",
    "R36 ① 상한":       "★ H3 의 오라클 시프트는 **상한이지 방법이 아니다**(TEST 적합)",
    "R40 ① λ ≠ 타당성":  "★★★ 교차레코드가 좋다고 **동작점이 고르다는 보장은 없다** — H5 가 잰다",
    "R40 ② 같은 통계":  "필요표본을 **관문 문턱 기준**으로",
}

CONFIG = dict(
    exp="quest46_q4e_operating_point", quest="ailab-2026-0046", step="operating-point",
    parent_exp=["quest46_q4d_calibrator_shift_equivariance"],
    purpose=("**동작점 최초 측정 + 축 결정.** (실무) Q3~Q4-D 는 전부 **문턱 없는 지표**였고 "
             "민감도·PPV·환자당 오경보를 **한 번도 안 쟀다**. (전략) 그리고 「매크로 0.58 이 "
             "무작위에 가깝다」는 인상은 **지표 오독**이다 — **PR-AUC 의 무작위 기저선은 "
             "0.5 가 아니라 그 레코드의 유병률**이다(유병률 0.02 면 lift 29배, 0.576 이면 "
             "1.0배). 유병률을 0.0070~0.5764 로 섞은 매크로 **하나로는 읽을 수 없으므로** "
             "H1 이 레코드별 유병률·PR-AUC·AUROC·lift 를 함께 낸다. ⚠️ 덧붙여 **퀘스트 안에서 "
             "지표가 갈렸다** — Q2/Q7-B′ 의 「매크로 0.8842」는 `roc_auc_score` 이고 Q4 라인의 "
             "「매크로 0.5796」은 `average_precision_score` 다. 같은 이름으로 불려 왔지만 "
             "**비교할 수 없는 수**이고, H1 이 같은 코호트에서 둘 다 내서 다리를 놓는다. "
             "★★★ **축 결정** — Q4-D 의 G0 가 이미 보였듯 레코드별 상수 로짓 시프트는 "
             "매크로를 **정확히** 불변으로 두는데(max|Δ| 0.0e+00) **층②가 바로 그 부류**다. "
             "즉 **층②로 할 수 있는 일에 매크로는 애초에 없다.** H2 가 임의의 무작위 "
             "시프트로 이걸 재확인하고, H3 이 **층②에 남은 천장**(교차레코드를 직접 "
             "최대화하는 오라클 시프트)을 재서 `TE` 가 거기 붙었는지 본다. 붙었으면 "
             "**층②는 끝난 것**이고 다음 축은 층①(표현) 또는 층④(코호트)다."),
    dataset="SVDB — svdb_data5.npz (리듬 특징만 · 새 데이터 0)",
    cals=list(CALS), primary_cal=PRIMARY_CAL, arms=list(ARMS), main=list(MAIN),
    flag_q=list(FLAG_Q), main_q=MAIN_Q, read_order=READ_ORDER,
    dev_every=DEV_EVERY, n_boot=NB_BOOT, n_perm=N_PERM, n_grid=N_GRID, smoke=SMOKE,
    ref=REF, rule_check=RULE_CHECK,
    predictions={
        "H0": "코호트 + **Platt 기울기 `a > 0`**. 음수면 순위가 뒤집히는데 팔이 다 같이 "
              "뒤집혀 Q4-D 의 G1 은 그대로 통과했다 — 미검사 결함이었다. 깨지면 **중단**",
        "H1": "★★★ **눈금(관문 아님)** — 레코드별 유병률·PR-AUC·AUROC·lift(AP/유병률). "
              "매크로 PR-AUC 의 무작위 기저선은 **평균 유병률**이다. 그리고 매크로 AUROC 를 "
              "같이 내서 Q2/Q7-B′ 의 0.8842(=AUROC)와 **비교 가능하게** 만든다",
        "H2": "★★★ **축(구성)** — 레코드마다 **무작위** 상수 로짓을 더해도 매크로가 "
              "**정확히** 불변인가. 층②(사전확률 정렬·부담 주입)가 이 부류이므로, "
              "성립하면 **층②는 매크로를 원리적으로 못 움직인다**. 깨지면 **중단**",
        "H3": "★★ **층②에 남은 천장(상한)** — 레코드별 시프트를 **교차레코드 AUROC 를 직접 "
              "최대화**하도록 좌표 상승으로 맞춘다(TEST 적합 = **상한이지 방법이 아니다** · "
              "R36 ①). `TE` 가 천장에 붙으면 층②는 끝난 것이다",
        "H4": "★★★ **동작점 A — 환자별 예산.** 각 레코드에서 점수 상위 q% 를 경보로 "
              "친다(문턱 없음 = 순수 순위). 레코드별 민감도·PPV. **매크로가 정하는 것**이 "
              "여기 나온다",
        "H5": "★★★ **동작점 B — 전역 단일 문턱.** 그 fold 의 **DEV 에서만** 전체 경보율이 "
              "q% 가 되는 문턱을 잡고 held-out 에 적용한다(R22·R34 ②). 레코드별 민감도·PPV 와 "
              "**환자 간 산포**. 교차레코드 AUROC 가 예측하는 「단일 문턱의 일관성」을 "
              "**직접** 검증한다. 주 관문 = `TE − A_em` 민감도(측정된 영점 상단 초과)",
        "H6": "결론 검산표"},
    caveat=("★★★ **H2 가 성립하면 이 퀘스트의 축이 정해진다** — 층②는 매크로를 못 움직이므로, "
            "매크로를 올리고 싶으면 **층①(표현)이나 층④(코호트)로 가야 한다**. 층②가 "
            "「실패」한 게 아니라 **애초에 그 일을 하는 도구가 아니다**. "
            "★★ **교차레코드가 좋다고 동작점이 고르다는 보장은 없다**(R40 ①) — AUROC 는 "
            "순위 지표이고 동작점은 문턱 지표다. H5 가 그 전환을 **직접** 잰다. "
            "★ **레코드 48(유병률 0.5764)에서는 PR-AUC 가 거의 무작위인 게 맞다**(관측 "
            "0.6805 · 기저선 0.5764 · lift 1.18배). 이건 지표 오독이 아니라 실제 한계다."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4e_operating_point", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4-E — 동작점, 그리고 축이 맞는가**")
run.log("  ★★★ (실무) 위양성·위음성을 **처음으로** 센다 — 지금까지 전부 문턱 없는 지표였다")
run.log("  ★★★ (전략) **층②는 매크로를 원리적으로 못 움직인다**(H2) — 축 결정이 걸려 있다")
run.log(f"  ★★ **PR-AUC 의 무작위 기저선은 0.5 가 아니라 유병률**이다 → H1 이 눈금을 세운다")
run.log(f"  ★ Q2/Q7-B′ 의 매크로 {REF['q7_macro_auroc']} 는 **AUROC** 다 — Q4 라인의 PR-AUC 와 "
        "비교할 수 없다. H1 이 다리를 놓는다")
run.log(f"  ★ 예산 FLAG_Q={FLAG_Q} 를 **사전 고정**(R34 ②) · 문턱은 **DEV 에서만**(R22)")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_PERM={N_PERM} · "
            f"N_GRID={N_GRID})")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【H-0】 코호트 · 보정기(★ 기울기 검사) · LORO
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【H-0】 코호트 · 보정기 · LORO")
run.log("=" * 100)
VERD, NOTE = {}, {}
def g_(k, v, d):
    VERD[k] = v; NOTE[k] = d; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; TT_ = (Y3[K] == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
RS = np.array(sorted(set(RID.tolist())))

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
RHY = np.nan_to_num(np.c_[_med - pre,
                          np.column_stack([1.0 - pre / (local_base(k) + 1e-9) for k in RHY_K]),
                          post - pre, np.nan_to_num(_std / (_mean + 1e-9)),
                          np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                    nan=0.0, posinf=0.0, neginf=0.0)

IDXS = {int(r): np.where(RID == r)[0] for r in RS}
REC_OK = [int(r) for r in RS
          if TT_[IDXS[int(r)]].sum() >= MIN_S and (~TT_[IDXS[int(r)]]).sum() >= MIN_N]
BURD = {r: float(TT_[IDXS[r]].mean()) for r in REC_OK}
s_all = np.array([int(TT_[IDXS[r]].sum()) for r in REC_OK], float)
DOMINANT = float(s_all.max() / s_all.sum()); DOM_REC = int(REC_OK[int(np.argmax(s_all))])
NRE = len(REC_OK)
MEAN_PREV = float(np.mean([BURD[r] for r in REC_OK]))
run.log(f"  레코드 {len(RS)} · 채점 가능 **{NRE}** · 유병률 "
        f"{min(BURD.values()):.4f}~{max(BURD.values()):.4f} · **평균 {MEAN_PREV:.4f}**")
run.log(f"  ★★ **평균 유병률 {MEAN_PREV:.4f} 이 매크로 PR-AUC 의 무작위 기저선**이다 "
        f"(매크로 AUROC 의 기저선은 0.5)")

EPS = 1e-6
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12)
    return np.log(p) - np.log1p(-p)

SLOPES = []
def make_cal(kind, s, y):
    """(확률, **보정 로짓**) — Platt 은 해석적으로 a·s+b(확률 왕복은 clip 에 걸린다)."""
    s = np.asarray(s, float); y = np.asarray(y).astype(int)
    if kind == "iso":
        ir = IsotonicRegression(out_of_bounds="clip", y_min=1e-6, y_max=1 - 1e-6).fit(s, y.astype(float))
        p_ = lambda v: np.clip(ir.predict(np.asarray(v, float)), EPS, 1 - EPS)
        return p_, (lambda v: logit(p_(v)))
    lr = LogisticRegression(max_iter=3000, C=1e6).fit(s.reshape(-1, 1), y)
    a, b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
    SLOPES.append(a)          # ★ Q4-D 미검사 결함 — 기울기를 모아 H0 에서 검사한다
    return (lambda v: 1.0 / (1.0 + np.exp(-(a * np.asarray(v, float) + b))),
            lambda v: a * np.asarray(v, float) + b)

def em_prior(p, pi_tr, iters=100, tol=1e-9, clip=1e-2):
    pi = float(pi_tr)
    for _ in range(int(iters)):
        w = pi / pi_tr; v = (1.0 - pi) / (1.0 - pi_tr)
        num = w * p
        pp = num / (num + v * (1.0 - p))
        new = float(np.clip(pp.mean(), clip, 1.0 - clip))
        if abs(new - pi) < tol:
            pi = new; break
        pi = new
    return pi

def split_rest(held):
    rest = sorted([r for r in REC_OK if r != held], key=lambda r: (BURD[r], r))
    dv = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
    return [r for r in rest if r not in set(dv)], dv

def loro(cal_kind, arm, y_override=None):
    """반환: (보정 로짓, **그 fold 의 DEV 로짓**) — 문턱을 DEV 에서만 잡기 위해."""
    out = np.full(len(K), np.nan)
    dev_pool = {}          # held -> (DEV 로짓, DEV 라벨)  ★ held-out 은 안 본다
    for held in REC_OK:
        tr_r, dv_r = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r]); dv = np.concatenate([IDXS[r] for r in dv_r])
        te = IDXS[held]
        use_b = arm in BSPEC
        if use_b:
            mu = float(np.mean([BURD[r] for r in tr_r]))
            bv = lambda ii: np.array([BURD[int(r)] for r in RID[ii]], float)
            Ftr = np.c_[RHY[tr], bv(tr)]
            feat = lambda ii, b_: np.c_[RHY[ii], b_]
        else:
            Ftr = RHY[tr]; feat = lambda ii, b_: RHY[ii]
        fmu, fsd = Ftr.mean(0), Ftr.std(0) + 1e-9
        ytr = TT_[tr].astype(int) if y_override is None else y_override[held]
        lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, ytr)
        sc = lambda ii, b_=None: lr.decision_function((feat(ii, b_) - fmu) / fsd)
        s_dv = sc(dv, np.array([BURD[int(r)] for r in RID[dv]], float)) if use_b else sc(dv)
        cp, cl = make_cal(cal_kind, s_dv, TT_[dv])
        pi_tr = float(TT_[dv].mean())
        if use_b:
            bh = BURD[held] if BSPEC[arm][1] == "true" else \
                 em_prior(cp(sc(te, np.full(len(te), mu))), pi_tr)
            l = cl(sc(te, np.full(len(te), bh)))
        else:
            l = cl(sc(te))
            if arm == "A_em":
                l = l + (logit(em_prior(cp(sc(te)), pi_tr)) - logit(pi_tr))
        out[te] = l
        dev_pool[held] = (cl(s_dv), TT_[dv].astype(int))
    return out, dev_pool

# ── 지표
def per_rec(L, fn):
    d = {}
    for r in REC_OK:
        pos = IDXS[r]; yy = TT_[pos].astype(int)
        if 0 < yy.sum() < len(yy) and np.all(np.isfinite(L[pos])):
            d[r] = float(fn(yy, L[pos]))
    return d
per_ap = lambda L: per_rec(L, average_precision_score)
per_auc = lambda L: per_rec(L, roc_auc_score)

def xrec_matrix(L):
    P = {r: np.sort(L[IDXS[r]][TT_[IDXS[r]]]) for r in REC_OK}
    N = {r: np.sort(L[IDXS[r]][~TT_[IDXS[r]]]) for r in REC_OK}
    M = np.full((NRE, NRE), np.nan)
    for a, ri in enumerate(REC_OK):
        p = P[ri]
        for b, rj in enumerate(REC_OK):
            if ri == rj or not len(p) or not len(N[rj]):
                continue
            q = N[rj]
            lo = np.searchsorted(q, p, "left"); hi = np.searchsorted(q, p, "right")
            M[a, b] = float((lo + 0.5 * (hi - lo)).sum() / (len(p) * len(q)))
    return M

def xrec_of(M):
    off = ~np.eye(NRE, dtype=bool)
    v = M[off & np.isfinite(M)]
    return float(v.mean()) if len(v) else float("nan")

def xrec_boot(Ms, seed, nb):
    rng = np.random.RandomState(seed); out = {k: [] for k in Ms}
    for _ in range(nb):
        idx = rng.randint(0, NRE, NRE); same = idx[:, None] == idx[None, :]
        for k, M in Ms.items():
            sub = M[np.ix_(idx, idx)]; m = (~same) & np.isfinite(sub)
            out[k].append(float(sub[m].mean()) if m.any() else float("nan"))
    return {k: np.asarray(v, float) for k, v in out.items()}

run.log("  LORO · 보정기 정의 완료 — 문턱용 **DEV 로짓**을 fold 마다 따로 남긴다(R22)")
CONFIG["cohort"] = dict(n_rec=len(RS), n_ok=NRE, dominant=DOMINANT, dom_rec=DOM_REC,
                        mean_prev=MEAN_PREV,
                        prev_min=min(BURD.values()), prev_max=max(BURD.values()))
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【H-A】 실행 · H0(기울기) · ★★★ H1 눈금 · ★★★ H2 축
run.log("\n" + "=" * 100)
run.log("【H-A】 실행 · H0(Platt 기울기) · H1(눈금) · H2(축)")
run.log("=" * 100)
T0 = time.time()
L, DEVP = {}, {}
for c in CALS:
    L[c], DEVP[c] = {}, {}
    for a in ARMS:
        L[c][a], DEVP[c][a] = loro(c, a)
    run.log(f"  ({time.time()-T0:>5.0f}초) 보정기 `{c}` — {len(ARMS)}팔 완료")

AP = {c: {a: per_ap(L[c][a]) for a in ARMS} for c in CALS}
AUC = {c: {a: per_auc(L[c][a]) for a in ARMS} for c in CALS}
MAC_AP = {c: {a: float(np.mean(list(AP[c][a].values()))) for a in ARMS} for c in CALS}
MAC_AUC = {c: {a: float(np.mean(list(AUC[c][a].values()))) for a in ARMS} for c in CALS}
XM = {c: {a: xrec_matrix(L[c][a]) for a in ARMS} for c in CALS}
XREC = {c: {a: xrec_of(XM[c][a]) for a in ARMS} for c in CALS}

# ── H0 — ★ Platt 기울기 (Q4-D 의 미검사 결함)
sl = np.array(SLOPES, float)
neg = int((sl <= 0).sum()); frac = neg / max(1, len(sl))
run.log(f"\n  H0 — Platt 기울기 {len(sl)}개 · 최소 **{sl.min():+.4f}** · 중앙 "
        f"{np.median(sl):+.4f} · **음수 {neg}개({frac:.1%})**")
# ★★★ 조용한 버그는 「하나가 뒤집히는 것」이 아니라 **「전부 뒤집히는 것」**이다.
#    하나가 뒤집히면 그 레코드의 AUROC 가 0.5 아래로 **보이므로** 증상이 드러난다.
#    그래서 중앙값 부호로 체계적 반전을 막고, 개별 퇴화는 **비율로** 제한한다.
#    (스모크 널 조건에서 44개 중 1개가 음수로 나왔다 — 실제로 생기는 일이다)
if np.median(sl) <= 0:
    raise AssetError(f"H0 실패 — Platt 기울기 **중앙값** {np.median(sl):.4f} ≤ 0. "
                     "체계적으로 순위가 뒤집혔다는 뜻이고, 팔이 다 같이 뒤집히면 항등 "
                     "검사(Q4-D 의 G1)는 그대로 통과한다 — 그래서 못 잡혔다(R29 ②)")
if frac > MAX_NEG_SLOPE:
    raise AssetError(f"H0 실패 — 기울기가 음수인 fold 가 {frac:.1%} > {MAX_NEG_SLOPE:.0%}. "
                     "그만큼의 레코드에서 모델이 무작위보다 나쁘다는 뜻이라 아래를 읽지 않는다")
g_("H0", "✅ 지지",
   f"Platt 기울기 중앙 {np.median(sl):+.4f} · 음수 {neg}/{len(sl)}({frac:.1%} ≤ "
   f"{MAX_NEG_SLOPE:.0%}) — 체계적 반전이 없다. ★ Q4-D 는 이 검사가 **없었다**"
   + ("" if neg == 0 else f". ⚠️ 음수 fold {neg}개는 그 레코드에서 모델이 무작위보다 "
                          "나쁘다는 **증상**이다 — H1 의 레코드별 AUROC 에서 0.5 아래로 보인다"))

# ── ★★★ H1 — 눈금. 유병률 없이는 PR-AUC 를 못 읽는다
run.log("\n  ★★★ H1 — **눈금**: PR-AUC 의 무작위 기저선은 0.5 가 아니라 **유병률**이다")
c = PRIMARY_CAL
LIFT = {a: {r: (AP[c][a][r] / BURD[r] if BURD[r] > 0 else float("nan"))
            for r in AP[c][a]} for a in ARMS}
run.log(f"  {'팔':<8}{'매크로 PR-AUC':>15}{'무작위 기저선':>14}{'매크로 lift':>13}"
        f"{'매크로 AUROC':>14}{'교차레코드':>12}")
for a in ARMS:
    lv = np.array(list(LIFT[a].values()), float)
    run.log(f"  {a:<8}{MAC_AP[c][a]:>15.4f}{MEAN_PREV:>14.4f}"
            f"{np.mean(lv[np.isfinite(lv)]):>12.1f}배{MAC_AUC[c][a]:>14.4f}{XREC[c][a]:>12.4f}")
run.log(f"  ▸ **매크로 AUROC 가 Q2/Q7-B′ 의 {REF['q7_macro_auroc']} 와 같은 지표**다 "
        f"(`roc_auc_score` · {REF['q7_note']}). 매크로 PR-AUC 와 **비교하면 안 된다**")
run.log(f"\n  유병률 사분위별 — 저유병률에서 lift 가 크고 고유병률에서 1 에 붙는다")
qs = np.quantile([BURD[r] for r in REC_OK], [0, .25, .5, .75, 1.0])
run.log(f"  {'유병률 구간':<22}{'n':>4}{'평균 유병률':>12}{'평균 AP':>10}{'lift':>9}{'AUROC':>9}")
QBAND = []
for i in range(4):
    lo_, hi_ = qs[i], qs[i + 1]
    rs = [r for r in AP[c]["TE"] if (lo_ <= BURD[r] <= hi_ if i == 3 else lo_ <= BURD[r] < hi_)]
    if not rs:
        continue
    mp = float(np.mean([BURD[r] for r in rs])); ma = float(np.mean([AP[c]["TE"][r] for r in rs]))
    QBAND.append(dict(lo=float(lo_), hi=float(hi_), n=len(rs), prev=mp, ap=ma,
                      lift=ma / mp, auc=float(np.mean([AUC[c]["TE"][r] for r in rs]))))
    run.log(f"  [{lo_:.4f}, {hi_:.4f}]{'':<4}{len(rs):>4}{mp:>12.4f}{ma:>10.4f}"
            f"{ma/mp:>8.1f}배{QBAND[-1]['auc']:>9.4f}")
run.log(f"  ★ 지배 레코드 {DOM_REC}(유병률 {BURD[DOM_REC]:.4f}) — AP "
        f"{AP[c]['TE'].get(DOM_REC, float('nan')):.4f} · lift "
        f"{LIFT['TE'].get(DOM_REC, float('nan')):.2f}배 · AUROC "
        f"{AUC[c]['TE'].get(DOM_REC, float('nan')):.4f}  ← **여기는 PR-AUC 가 거의 무작위가 맞다**")
g_("H1", "(관문 아님)",
   f"매크로 PR-AUC {MAC_AP[c]['TE']:.4f} 의 무작위 기저선은 **{MEAN_PREV:.4f}**(평균 lift "
   f"{np.mean([v for v in LIFT['TE'].values() if np.isfinite(v)]):.1f}배)이고, 같은 코호트의 "
   f"매크로 **AUROC 는 {MAC_AUC[c]['TE']:.4f}** 다 — Q7 라인의 {REF['q7_macro_auroc']} 와 "
   f"비교할 수 있는 건 **이쪽**이다")

# ── ★★★ H2 — 축. 층②(레코드별 상수 시프트)는 매크로를 원리적으로 못 움직인다
run.log("\n  ★★★ H2 — **축**: 레코드마다 **무작위** 상수 로짓을 더해도 매크로가 불변인가")
rng_h2 = np.random.RandomState(SEED0 + 900)
worst = 0.0
for trial in range(5):
    sh = {r: float(rng_h2.normal(0, 3.0)) for r in REC_OK}
    Ls = L[c]["TE"].copy()
    for r in REC_OK:
        Ls[IDXS[r]] += sh[r]
    d_ap = max(abs(per_ap(Ls)[r] - AP[c]["TE"][r]) for r in AP[c]["TE"])
    d_auc = max(abs(per_auc(Ls)[r] - AUC[c]["TE"][r]) for r in AUC[c]["TE"])
    worst = max(worst, d_ap, d_auc)
    run.log(f"    시행 {trial+1} — 시프트 SD 3.0 · 레코드별 max|ΔAP| {d_ap:.2e} · "
            f"max|ΔAUROC| {d_auc:.2e}")
if worst >= TOL_IDENT:
    raise AssetError(f"H2 실패({worst:.3e}) — 레코드 내 지표는 레코드별 상수 시프트에 "
                     "불변이어야 한다. 깨지면 구현이 틀렸다(R29 ②)")
g_("H2", "✅ 지지",
   f"**임의의** 레코드별 상수 시프트에서 레코드 내 지표가 정확히 불변이다({worst:.1e}) — "
   "★★★ **층②(사전확률 정렬·부담 주입)가 바로 그 부류이므로, 층②로 매크로를 올리는 것은 "
   "원리적으로 불가능하다.** 층②가 실패한 게 아니라 **그 일을 하는 도구가 아니다**")
CONFIG["H0"] = dict(platt_slope_min=float(sl.min()), platt_slope_med=float(np.median(sl)),
                    n=len(sl), n_neg=neg, frac_neg=float(frac), max_neg=MAX_NEG_SLOPE)
CONFIG["H1"] = dict(macro_ap=MAC_AP, macro_auc=MAC_AUC, xrec=XREC, mean_prev=MEAN_PREV,
                    lift={a: float(np.mean([v for v in LIFT[a].values() if np.isfinite(v)]))
                          for a in ARMS}, qband=QBAND,
                    per_rec_prev={r: BURD[r] for r in REC_OK},
                    per_rec_ap=AP[PRIMARY_CAL], per_rec_auc=AUC[PRIMARY_CAL])
CONFIG["H2"] = dict(worst=float(worst), tol=TOL_IDENT, trials=5)
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【H-B】 ★★ H3 — 층②에 **남은** 천장
run.log("\n" + "=" * 100)
run.log("【H-B】 ★★ H3 — 층②에 남은 천장(교차레코드를 직접 최대화하는 오라클 시프트)")
run.log("=" * 100)
run.log("  ⚠️ **TEST 에서 적합한다 — 상한이지 방법이 아니다**(R36 ①). 층②가 아무리 잘해도")
run.log("     여기까지라는 선을 긋는 용도다.")
T3 = time.time()
c = PRIMARY_CAL

def _pn(L):
    return ({r: np.sort(L[IDXS[r]][TT_[IDXS[r]]]) for r in REC_OK},
            {r: np.sort(L[IDXS[r]][~TT_[IDXS[r]]]) for r in REC_OK})

def _auc_pair(p, q):
    if not len(p) or not len(q):
        return np.nan
    lo = np.searchsorted(q, p, "left"); hi = np.searchsorted(q, p, "right")
    return float((lo + 0.5 * (hi - lo)).sum() / (len(p) * len(q)))

def xrec_opt(L, n_grid=N_GRID, span=4.0, passes=2, seed=0):
    """좌표 상승 — 레코드별 시프트를 하나씩 격자 탐색한다."""
    P, N = _pn(L)
    sh = {r: 0.0 for r in REC_OK}
    M = np.array([[_auc_pair(P[ri], N[rj]) if ri != rj else np.nan
                   for rj in REC_OK] for ri in REC_OK], float)
    cur = xrec_of(M)
    grid = np.linspace(-span, span, n_grid)
    for _ in range(passes):
        for a, ri in enumerate(REC_OK):
            best, bc = cur, 0.0
            base_row = M[a, :].copy(); base_col = M[:, a].copy()
            for gcand in grid:
                if abs(gcand) < 1e-12:
                    continue
                p2 = P[ri] + gcand; n2 = N[ri] + gcand
                row = np.array([_auc_pair(p2, N[rj]) if rj != ri else np.nan
                                for rj in REC_OK], float)
                col = np.array([_auc_pair(P[rj], n2) if rj != ri else np.nan
                                for rj in REC_OK], float)
                M[a, :] = row; M[:, a] = col
                v = xrec_of(M)
                if v > best:
                    best, bc = v, gcand
                M[a, :] = base_row; M[:, a] = base_col
            if bc != 0.0:
                P[ri] = P[ri] + bc; N[ri] = N[ri] + bc; sh[ri] += bc
                M[a, :] = np.array([_auc_pair(P[ri], N[rj]) if rj != ri else np.nan
                                    for rj in REC_OK], float)
                M[:, a] = np.array([_auc_pair(P[rj], N[ri]) if rj != ri else np.nan
                                    for rj in REC_OK], float)
                cur = best
    return cur, sh

# ★★★ 천장을 **두 모델에서** 잰다. `A_em` 은 raw 의 층② 시프트이므로 **raw-천장**과
#     견주는 게 사과 대 사과다. `TE` 는 층② 가 아니라 **다른 모델**(부담을 특징으로)이라
#     raw-천장을 넘어도 정당하고, 넘으면 그게 곧 **「모델을 바꾸는 게 층②보다 낫다」**는
#     증거다.
CEIL, SH_OPT = xrec_opt(L[c]["raw"])
CEIL_TE, _SH_TE = xrec_opt(L[c]["TE"])
run.log(f"  ({time.time()-T3:.0f}초) 좌표 상승 완료 — 격자 {N_GRID}점 × 2회 × 2모델")
run.log(f"\n  {'':<28}{'교차레코드':>12}{'raw-천장 대비':>16}")
for a in ARMS:
    run.log(f"  {a:<28}{XREC[c][a]:>12.4f}{XREC[c][a] - CEIL:>15.4f}")
run.log(f"  {'★ raw + 오라클 시프트(상한)':<28}{CEIL:>12.4f}{0.0:>15.4f}")
run.log(f"  {'★ TE  + 오라클 시프트(상한)':<28}{CEIL_TE:>12.4f}{CEIL_TE - CEIL:>15.4f}")
GAP = CEIL - XREC[c]["A_em"]          # ★ 층② 의 실제 방법(A_em)이 자기 천장까지 남긴 폭
GAP_TE = CEIL_TE - XREC[c]["TE"]
run.log(f"\n  ★★ **층② 의 방법(`A_em`)이 raw-천장까지 남긴 폭 = {GAP:+.4f}** "
        f"(천장 {CEIL:.4f} · A_em {XREC[c]['A_em']:.4f})")
run.log(f"  ★★ `TE`(모델을 바꾼 쪽) {XREC[c]['TE']:.4f} — raw-천장 대비 "
        f"**{XREC[c]['TE'] - CEIL:+.4f}**"
        + ("  ← ★★★ **층② 가 raw 에서 낼 수 있는 최대치를 넘었다**"
           if XREC[c]["TE"] > CEIL else "  (아직 raw-천장 안이다)"))
run.log(f"  ★ TE 위에 다시 층② 를 얹은 천장 {CEIL_TE:.4f} — TE 가 거기까지 남긴 폭 {GAP_TE:+.4f}")
run.log(f"  ▸ 오라클 시프트의 크기 — 중앙 {np.median([abs(v) for v in SH_OPT.values()]):.3f} · "
        f"최대 {max(abs(v) for v in SH_OPT.values()):.3f} 로짓")
_rho_shift = np.corrcoef([SH_OPT[r] for r in REC_OK],
                         [logit(BURD[r]) for r in REC_OK])[0, 1]
run.log(f"  ▸ 최적 시프트와 **logit(유병률)** 의 상관 = **{_rho_shift:+.4f}** — "
        + ("사전확률 정렬이 대체로 옳은 방향이다" if _rho_shift > 0.3 else
           "★ **최적 시프트는 유병률이 아니다**(Q4-D 에서 `TE`(추정) > `TT`(오라클) 였던 이유)"))
g_("H3", "(관문 아님)",
   f"raw-천장 **{CEIL:.4f}** · 층② 의 방법 `A_em` {XREC[c]['A_em']:.4f}(남은 폭 {GAP:+.4f}) · "
   f"모델을 바꾼 `TE` {XREC[c]['TE']:.4f}(raw-천장 대비 {XREC[c]['TE'] - CEIL:+.4f}). "
   + ("★★★ **모델 교체가 층② 의 최대치를 넘는다 — 다음 축은 층①/층④다**"
      if XREC[c]["TE"] > CEIL else
      f"★ 아직 raw-천장 안이다 — 층② 에 {GAP:.4f} 가 남아 있다(상한이지 방법이 아니다)"))
CONFIG["H3"] = dict(ceiling=float(CEIL), ceiling_TE=float(CEIL_TE),
                    gap_Aem=float(GAP), gap_TE=float(GAP_TE),
                    TE_minus_ceiling=float(XREC[c]["TE"] - CEIL),
                    shift_med=float(np.median([abs(v) for v in SH_OPT.values()])),
                    shift_max=float(max(abs(v) for v in SH_OPT.values())),
                    rho_with_logit_prev=float(_rho_shift),
                    xrec={a: XREC[c][a] for a in ARMS})
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【H-C】 ★★★ H4 환자별 예산 · H5 전역 단일 문턱 (위양성·위음성)
run.log("\n" + "=" * 100)
run.log("【H-C】 ★★★ H4(환자별 예산) · H5(전역 단일 문턱) — 위양성·위음성을 **처음으로** 센다")
run.log("=" * 100)
c = PRIMARY_CAL

def budget_stats(L, q):
    """★ 환자별 예산 — 각 레코드에서 상위 q% 를 경보로 친다(문턱 없음 = 순수 순위)."""
    out = {}
    for r in REC_OK:
        pos = IDXS[r]; sc = L[pos]; yy = TT_[pos]
        k = max(1, int(round(q * len(pos))))
        thr = np.partition(sc, -k)[-k]
        fl = sc >= thr
        tp = int((fl & yy).sum()); fp = int((fl & ~yy).sum()); fn = int((~fl & yy).sum())
        out[r] = dict(sens=tp / max(1, tp + fn), ppv=tp / max(1, tp + fp),
                      tp=tp, fp=fp, fn=fn, flagged=int(fl.sum()), n=len(pos),
                      prev=BURD[r])
    return out

def global_stats(L, devp, q):
    """★★ 전역 단일 문턱 — 그 fold 의 **DEV 에서만** 경보율 q 가 되는 문턱을 잡는다(R22)."""
    out = {}
    for r in REC_OK:
        dl, _dy = devp[r]
        thr = float(np.quantile(dl, 1.0 - q))     # ← held-out 은 안 본다
        pos = IDXS[r]; sc = L[pos]; yy = TT_[pos]
        fl = sc >= thr
        tp = int((fl & yy).sum()); fp = int((fl & ~yy).sum()); fn = int((~fl & yy).sum())
        out[r] = dict(sens=tp / max(1, tp + fn), ppv=(tp / fl.sum()) if fl.sum() else np.nan,
                      tp=tp, fp=fp, fn=fn, flagged=int(fl.sum()), n=len(pos),
                      rate=float(fl.mean()), thr=thr, prev=BURD[r])
    return out

BUD = {q: {a: budget_stats(L[c][a], q) for a in ARMS} for q in FLAG_Q}
GLB = {q: {a: global_stats(L[c][a], DEVP[c][a], q) for a in ARMS} for q in FLAG_Q}

def summarize(d, key):
    v = np.array([d[r][key] for r in REC_OK if np.isfinite(d[r][key])], float)
    return v

for q in FLAG_Q:
    run.log(f"\n  ── 예산 q = **{q:.0%}**")
    run.log(f"  {'':<24}{'민감도 평균':>12}{'SD':>8}{'최소':>8}{'PPV 평균':>10}"
            f"{'경보율 평균':>12}{'SD':>8}")
    for nm, D in (("H4 환자별 예산", BUD[q]), ("H5 전역 단일 문턱", GLB[q])):
        for a in ARMS:
            s_ = summarize(D[a], "sens"); p_ = summarize(D[a], "ppv")
            rt = (summarize(D[a], "rate") if nm.startswith("H5")
                  else np.full(len(s_), q))
            run.log(f"  {nm[:2] + ' ' + a:<24}{s_.mean():>12.4f}{s_.std(ddof=1):>8.4f}"
                    f"{s_.min():>8.4f}{np.nanmean(p_):>10.4f}{rt.mean():>12.4f}"
                    f"{rt.std(ddof=1):>8.4f}")

run.log(f"\n  ★★ **H4 는 문턱이 없다**(환자마다 상위 {MAIN_Q:.0%}) → 경보율 산포 0. "
        "매크로가 정하는 것이 그대로 나온다")
run.log("  ★★ **H5 는 단일 문턱**이라 경보율이 환자마다 달라진다 → 그 산포가 곧 "
        "**교차레코드 정렬의 실무적 대가**다")

# ── ★★★ 주 관문 — **경보율 이탈**(|실현 경보율 − 목표|). 낮을수록 좋다.
#    ⚠️ 스모크가 잡았다: 서로 다른 동작점에서 **민감도를 비교하면 안 된다**.
#       `A_em` 민감도 0.2575 가 `TE` 0.1496 보다 높아 보였지만 경보율이 **0.2505**
#       (목표 5% 의 5배)였다 — 많이 울려서 많이 맞힌 것뿐이다.
#    ▸ **순위 품질**은 H4(환자별 예산 = 경보율이 구성으로 일치)에서 비교한다
#    ▸ **문턱 전이**는 여기서 경보율 이탈로 비교한다 — 이게 「단일 문턱이 통하나」다
q = MAIN_Q
for D in (GLB[q_] for q_ in FLAG_Q):
    pass
for q_ in FLAG_Q:
    for a in ARMS:
        for r in REC_OK:
            GLB[q_][a][r]["dev"] = abs(GLB[q_][a][r]["rate"] - q_)
ks = [r for r in REC_OK if np.isfinite(GLB[q][MAIN[0]][r]["sens"])
      and np.isfinite(GLB[q][MAIN[1]][r]["sens"])]
H5 = dict()
for key, seed in (("dev", SEED0 + 41), ("sens", SEED0 + 42), ("ppv", SEED0 + 43),
                  ("rate", SEED0 + 44)):
    aa = [GLB[q][MAIN[0]][r][key] for r in ks]; bb = [GLB[q][MAIN[1]][r][key] for r in ks]
    m_, lo_, hi_, n_ = boot_pair(aa, bb, seed, NB_BOOT)
    sd_ = boot_sd_diff(aa, bb, seed + 100, NB_BOOT)
    H5[key] = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)),
                   sd_diff=sd_[0], sd_lo=sd_[1], sd_hi=sd_[2])
# H4 — 경보율이 **구성으로 일치**하므로 민감도 비교가 공정하다
kb = [r for r in REC_OK if np.isfinite(BUD[q][MAIN[0]][r]["sens"])
      and np.isfinite(BUD[q][MAIN[1]][r]["sens"])]
m4, lo4, hi4, n4 = boot_pair([BUD[q][MAIN[0]][r]["sens"] for r in kb],
                             [BUD[q][MAIN[1]][r]["sens"] for r in kb], SEED0 + 51, NB_BOOT)
H4D = dict(mean=m4, lo=lo4, hi=hi4, n=int(n4), mde=float(mde(lo4, hi4)))
run.log(f"\n  주 대비 `{MAIN[1]} − {MAIN[0]}` (q={q:.0%})")
run.log(f"    **H4 환자별 예산 · 민감도** Δ {m4:>+8.4f} [{lo4:+.4f}, {hi4:+.4f}] "
        "← 경보율이 **구성으로 일치**하므로 이게 **순위 품질**의 공정한 비교다")
for key, tag in (("dev", "★★★ **주 관문** — 낮을수록 좋다(단일 문턱이 통한다)"),
                 ("rate", "실현 경보율"), ("sens", "⚠️ 동작점이 다르면 비교 불가"),
                 ("ppv", "")):
    h = H5[key]
    run.log(f"    H5 {key:<5} Δ {h['mean']:>+8.4f} [{h['lo']:+.4f}, {h['hi']:+.4f}] · "
            f"산포 Δ {h['sd_diff']:>+8.4f} [{h['sd_lo']:+.4f}, {h['sd_hi']:+.4f}]  {tag}")

# ── ★★ 대비의 영점
run.log(f"\n  ★★ **대비의 영점** — 학습 라벨 치환 (reps={N_PERM})")
nul = {}
for s_ in range(N_PERM):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    yov = {}
    for held in REC_OK:
        tr_r, _ = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        yov[held] = TT_[tr].astype(int)[rr.permutation(len(tr))]
    la, dpa = loro(c, MAIN[0], y_override=yov)
    lb, dpb = loro(c, MAIN[1], y_override=yov)
    ga, gb = global_stats(la, dpa, q), global_stats(lb, dpb, q)
    for r in REC_OK:
        if np.isfinite(ga[r]["rate"]) and np.isfinite(gb[r]["rate"]):
            nul.setdefault(r, []).append(abs(gb[r]["rate"] - q) - abs(ga[r]["rate"] - q))
    run.log(f"    ({time.time()-T3:>5.0f}초) 영점 rep {s_+1}/{N_PERM}")
NS = boot_mean([float(np.mean(v)) for v in nul.values()], SEED0 + 61, NB_BOOT)
run.log(f"    경보율 이탈 영점 **{NS[0]:+.4f}** [{NS[1]:+.4f}, {NS[2]:+.4f}] (레코드 {NS[3]})")
# ★ 「낮을수록 좋다」 방향이므로 문턱은 **min(0, 영점 하단)** 이다
H5_THR = min(0.0, NS[1]) if np.isfinite(NS[1]) else float("nan")
run.log(f"    ▸ 문턱 = min(0, 영점 하단) **{H5_THR:+.4f}** (방향 `<` — 이탈이 작아야 한다)")
h5v = decide(H5["dev"]["lo"], H5["dev"]["hi"], H5_THR, "<")
g_("H5", h5v,
   f"**경보율 이탈** Δ {H5['dev']['mean']:+.4f} [{H5['dev']['lo']:+.4f}, "
   f"{H5['dev']['hi']:+.4f}] · 산포 {H5['dev']['sd_diff']:+.4f} · 문턱 {H5_THR:+.4f} — "
   + ("**`TE` 가 단일 문턱을 더 고르게 만든다**" if h5v.startswith("✅") else
      ("`TE` 가 오히려 더 어긋난다" if h5v.startswith("❌") else "가르지 못했다(R33 ①)"))
   + f"  |  참고 민감도 Δ {H5['sens']['mean']:+.4f}(동작점이 달라 직접 비교 불가) · "
     f"H4 순위 Δ {H4D['mean']:+.4f}"
   + ("" if np.isfinite(H5_THR) else " — ★ **영점 미측정**(R26)"))
g_("H4", "(관문 아님)",
   f"환자별 예산 q={MAIN_Q:.0%} — `TE` 민감도 평균 "
   f"{summarize(BUD[MAIN_Q]['TE'], 'sens').mean():.4f} (SD "
   f"{summarize(BUD[MAIN_Q]['TE'], 'sens').std(ddof=1):.4f} · 최소 "
   f"{summarize(BUD[MAIN_Q]['TE'], 'sens').min():.4f}) · PPV "
   f"{np.nanmean(summarize(BUD[MAIN_Q]['TE'], 'ppv')):.4f}")
CONFIG["H4"] = {f"q{int(q_*100)}": {a: {str(r): BUD[q_][a][r] for r in REC_OK}
                                    for a in ARMS} for q_ in FLAG_Q}
CONFIG["H5"] = dict(diff=H5, null=dict(mean=NS[0], lo=NS[1], hi=NS[2], n=int(NS[3])),
                    thr=float(H5_THR), h4_sens_diff=H4D,
                    summary={f"q{int(q_*100)}": {a: dict(
                        sens_mean=float(summarize(GLB[q_][a], "sens").mean()),
                        sens_sd=float(summarize(GLB[q_][a], "sens").std(ddof=1)),
                        sens_min=float(summarize(GLB[q_][a], "sens").min()),
                        ppv_mean=float(np.nanmean(summarize(GLB[q_][a], "ppv"))),
                        rate_mean=float(summarize(GLB[q_][a], "rate").mean()),
                        rate_sd=float(summarize(GLB[q_][a], "rate").std(ddof=1)),
                        dev_mean=float(summarize(GLB[q_][a], "dev").mean()))
                        for a in ARMS} for q_ in FLAG_Q})
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【H-D】 필요표본 · 진단 · ★ H6 검산표
run.log("\n" + "=" * 100)
run.log("【H-D】 필요표본 · 진단 · H6 결론 검산표")
run.log("=" * 100)
c = PRIMARY_CAL
run.log(f"  필요표본 (**관문 문턱 기준** · 레코드 · 현재 {NRE})")
eff = H5["dev"]["mean"] - H5_THR
n5 = need_super(NRE, H5["dev"]["mde"], eff); n8 = need_super(NRE, H5["dev"]["mde"], eff, True)
bad = (not np.isfinite(eff)) or abs(eff) < H5["dev"]["mde"]
run.log(f"  H5 경보율이탈 효과-문턱 {eff:+.4f} · 반폭 {H5['sens']['mde']:.4f} · "
        f"n(50%) {n5:.0f} · n(80%) {n8:.0f}  "
        + ("★ **해석 불가**(R41 ②)" if bad else
           ("이미 충분하다" if n8 <= NRE else "표본이 더 필요하다")))

run.log(f"\n  ★ 위양성·위음성 — **전역 단일 문턱 q={MAIN_Q:.0%}** 에서 최악 레코드 5개")
G = GLB[MAIN_Q]["TE"]
worst5 = sorted(REC_OK, key=lambda r: G[r]["sens"])[:5]
run.log(f"  {'레코드':<8}{'유병률':>9}{'민감도':>9}{'PPV':>9}{'FP':>7}{'FN':>7}{'경보율':>9}")
for r in worst5:
    run.log(f"  #{r:<7}{G[r]['prev']:>9.4f}{G[r]['sens']:>9.4f}"
            f"{(G[r]['ppv'] if np.isfinite(G[r]['ppv']) else float('nan')):>9.4f}"
            f"{G[r]['fp']:>7d}{G[r]['fn']:>7d}{G[r]['rate']:>9.4f}")
run.log(f"  ★ 지배 레코드 {DOM_REC} — 민감도 {G[DOM_REC]['sens']:.4f} · PPV "
        f"{G[DOM_REC]['ppv']:.4f} · FP {G[DOM_REC]['fp']} · FN {G[DOM_REC]['fn']}")

run.log("\n  ★ H6 — **결론 검산표**")
CHECK = [
    dict(claim=f"H0 — Platt 기울기에 **체계적 반전이 없다**(중앙 "
               f"{CONFIG['H0']['platt_slope_med']:+.4f} · 최소 "
               f"{CONFIG['H0']['platt_slope_min']:+.4f})",
         num=f"fold×팔 {CONFIG['H0']['n']}개 전수 · 음수 {CONFIG['H0']['n_neg']}개"
             f"({CONFIG['H0']['frac_neg']:.1%} ≤ {MAX_NEG_SLOPE:.0%}) · 중앙 "
             f"{CONFIG['H0']['platt_slope_med']:+.4f}",
         assume="**없음** — 런타임 검사",
         iffalse="★★ 조용한 버그는 「하나가 뒤집히는 것」이 아니라 **「전부 뒤집히는 것」**"
                 "이다 — 하나면 그 레코드 AUROC 가 0.5 아래로 보여 증상이 난다. 전부면 "
                 "팔이 다 같이 뒤집혀 **항등 검사(Q4-D 의 G1)는 그대로 통과**한다"),
    dict(claim=f"H1 눈금 — 매크로 PR-AUC {MAC_AP[c]['TE']:.4f} 의 무작위 기저선은 "
               f"**{MEAN_PREV:.4f}**(평균 lift {CONFIG['H1']['lift']['TE']:.1f}배)",
         num=f"같은 코호트 매크로 **AUROC {MAC_AUC[c]['TE']:.4f}** — Q2/Q7-B′ 의 "
             f"{REF['q7_macro_auroc']} 와 **이쪽**이 같은 지표다({REF['q7_note']})",
         assume="**없음** — 두 지표를 같은 점수에서 냈다",
         iffalse="★★★ PR-AUC 를 0.5 기준으로 읽으면 「무작위에 가깝다」로 오독한다. "
                 f"단 유병률 {BURD[DOM_REC]:.4f} 인 레코드 {DOM_REC} 는 **실제로** "
                 f"lift {CONFIG['H1']['lift']['TE'] and AP[c]['TE'][DOM_REC]/BURD[DOM_REC]:.2f}배다"),
    dict(claim=f"H2 축 — 임의의 레코드별 상수 시프트에서 레코드 내 지표가 불변"
               f"({CONFIG['H2']['worst']:.1e}) → {VERD['H2']}",
         num="시프트 SD 3.0 로짓 · 5회 시행 · PR-AUC 와 AUROC 둘 다",
         assume="**없음** — 구성이고 런타임 검사한다",
         iffalse="★★★ **이것이 축 결정의 근거다** — 층②(사전확률 정렬·부담 주입)가 이 "
                 "부류이므로 **층②로 매크로를 올리는 것은 원리적으로 불가능**하다"),
    dict(claim=f"H3 층②의 천장 {CONFIG['H3']['ceiling']:.4f} · `TE` {XREC[c]['TE']:.4f} · "
               f"남은 폭 {CONFIG['H3']['gap_TE']:+.4f}",
         num=f"좌표 상승 · 격자 {N_GRID}점 × 2회 · 최적 시프트와 logit(유병률)의 상관 "
             f"{CONFIG['H3']['rho_with_logit_prev']:+.4f}",
         assume="**TEST 에서 적합한다** — 상한이지 방법이 아니다(R36 ①)",
         iffalse="★ 남은 폭이 좁으면 **층②는 끝난 것**이고 다음 축은 층①/층④다"),
    dict(claim=f"H5 단일 문턱 **경보율 이탈** {H5['dev']['mean']:+.4f} "
               f"[{H5['dev']['lo']:+.4f}, {H5['dev']['hi']:+.4f}] → {VERD['H5']}",
         num=f"영점 {CONFIG['H5']['null']['mean']:+.4f} · 문턱 {H5_THR:+.4f} · "
             f"산포 Δ {H5['dev']['sd_diff']:+.4f} · ⚠️ 민감도는 동작점이 달라 "
             f"직접 비교 불가(A_em 경보율이 목표를 크게 벗어난다)",
         assume="문턱을 **그 fold 의 DEV 에서만** 잡았다(R22 · R34 ②)",
         iffalse="★★ 교차레코드 AUROC 가 좋아도 **동작점이 고르다는 보장은 없다**(R40 ①) — "
                 "그래서 직접 쟀다"),
    dict(claim=f"예산 FLAG_Q={FLAG_Q} 를 **사전 고정**했다",
         num="TEST 에서 쓸어보지 않았다",
         assume="**없음**",
         iffalse="★ 문턱을 TEST 에서 고르면 어떤 수든 만들 수 있다(R34 ②)"),
]
for i, ck in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{ck['claim']}**")
    run.log(f"      근거   {ck['num']}")
    run.log(f"      가정   {ck['assume']}")
    run.log(f"      틀리면 {ck['iffalse']}")
CONFIG["need"] = dict(h5_sens=dict(effect=float(eff), half=float(H5["sens"]["mde"]),
                                   sup50=float(n5), sup80=float(n8),
                                   uninterpretable=bool(bad)))
CONFIG["H6"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【H-E】 그림 · 요약 · 마무리
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
c = PRIMARY_CAL
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

pv = np.array([BURD[r] for r in REC_OK]); apv = np.array([AP[c]["TE"][r] for r in REC_OK])
ax[0].scatter(pv, apv, s=34, color="tab:blue", label="TE  PR-AUC")
ax[0].plot([0, pv.max()], [0, pv.max()], "k--", lw=1.0, label="chance (= prevalence)")
ax[0].scatter(pv, [AUC[c]["TE"][r] for r in REC_OK], s=18, color="tab:orange",
              marker="^", label="TE  AUROC")
ax[0].axhline(0.5, ls=":", color="tab:gray", lw=1.0)
ax[0].annotate(f"#{DOM_REC}", (BURD[DOM_REC], AP[c]["TE"][DOM_REC]), fontsize=8,
               xytext=(5, -10), textcoords="offset points")
ax[0].set_xlabel("record prevalence"); ax[0].set_ylabel("score")
ax[0].set_title("H1 : PR-AUC chance line IS prevalence", fontsize=9)
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)

xs = np.arange(len(ARMS))
ax[1].bar(xs - 0.2, [XREC[c][a] for a in ARMS], width=0.4, color="tab:orange",
          label="cross-record")
ax[1].axhline(CEIL, ls="--", color="tab:red", lw=1.2, label=f"layer-2 ceiling {CEIL:.3f}")
ax[1].axhline(0.5, ls=":", color="k", lw=1.0, label="chance 0.5")
ax[1].bar(xs + 0.2, [MAC_AP[c][a] for a in ARMS], width=0.4, color="tab:green",
          label="macro PR-AUC")
ax[1].axhline(MEAN_PREV, ls=":", color="tab:green", lw=1.2,
              label=f"PR chance {MEAN_PREV:.3f}")
ax[1].set_xticks(xs); ax[1].set_xticklabels(ARMS, fontsize=8)
ax[1].set_title("H3 : how much room is left in layer 2", fontsize=9)
ax[1].legend(fontsize=6); ax[1].grid(alpha=.3, axis="y")

for a, col, mk in (("A_em", "tab:gray", "o"), ("TE", "tab:purple", "^")):
    ax[2].scatter([BURD[r] for r in REC_OK],
                  [GLB[MAIN_Q][a][r]["sens"] for r in REC_OK], s=28, color=col,
                  marker=mk, label=f"{a} (global thr)")
ax[2].scatter([BURD[r] for r in REC_OK],
              [BUD[MAIN_Q]["TE"][r]["sens"] for r in REC_OK], s=16, color="tab:cyan",
              marker="s", label="TE (per-patient budget)")
ax[2].set_xlabel("record prevalence"); ax[2].set_ylabel(f"sensitivity @ {MAIN_Q:.0%}")
ax[2].set_title("H4/H5 : does one threshold work for everyone", fontsize=9)
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q4e_operating_point", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:6]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
run.log(f"  ★★★ **눈금** — 매크로 PR-AUC {MAC_AP[c]['TE']:.4f} (무작위 기저선 "
        f"**{MEAN_PREV:.4f}** · 평균 lift {CONFIG['H1']['lift']['TE']:.1f}배) · "
        f"매크로 **AUROC {MAC_AUC[c]['TE']:.4f}** · 교차레코드 {XREC[c]['TE']:.4f}")
run.log(f"     ▸ Q2/Q7-B′ 의 {REF['q7_macro_auroc']} 와 비교 가능한 건 **AUROC** 쪽이다")
run.log(f"     ▸ 지배 레코드 {DOM_REC}(유병률 {BURD[DOM_REC]:.4f}) — AP "
        f"{AP[c]['TE'][DOM_REC]:.4f} · lift {AP[c]['TE'][DOM_REC]/BURD[DOM_REC]:.2f}배 "
        f"← **여기는 거의 무작위가 맞다**")
run.log("")
run.log(f"  ★★★ **축** — {VERD['H2']} 임의 시프트에서 레코드 내 지표 불변"
        f"({CONFIG['H2']['worst']:.1e})")
if CONFIG["H3"]["gap_TE"] < 0.02:
    run.log(f"     그리고 층②의 천장 {CEIL:.4f} 에 `TE` {XREC[c]['TE']:.4f} 가 붙었다"
            f"(남은 폭 {CONFIG['H3']['gap_TE']:+.4f})")
    run.log("     → ★★★ **층②는 끝났다. 다음 축은 층①(표현) 또는 층④(코호트)다.**")
else:
    run.log(f"     층②의 천장 {CEIL:.4f} · `TE` {XREC[c]['TE']:.4f} · 남은 폭 "
            f"{CONFIG['H3']['gap_TE']:+.4f} → 층②에 아직 여지가 있다")
    run.log(f"     ▸ 최적 시프트와 logit(유병률)의 상관 "
            f"{CONFIG['H3']['rho_with_logit_prev']:+.4f}")
run.log("")
S = CONFIG["H5"]["summary"][f"q{int(MAIN_Q*100)}"]
run.log(f"  ★★★ **위양성·위음성**(전역 단일 문턱 q={MAIN_Q:.0%})")
for a in ARMS:
    run.log(f"     {a:<8} 민감도 {S[a]['sens_mean']:.4f} (SD {S[a]['sens_sd']:.4f} · 최소 "
            f"{S[a]['sens_min']:.4f}) · PPV {S[a]['ppv_mean']:.4f} · 경보율 "
            f"{S[a]['rate_mean']:.4f} (SD {S[a]['rate_sd']:.4f}) · **이탈 "
            f"{S[a]['dev_mean']:.4f}**")
run.log(f"     주 관문 `{MAIN[1]} − {MAIN[0]}` — {VERD['H5']}")
Sb = summarize(BUD[MAIN_Q]["TE"], "sens")
run.log(f"     ▸ **환자별 예산**이면 같은 팔이 민감도 {Sb.mean():.4f} (SD {Sb.std(ddof=1):.4f}) "
        "— 단일 문턱과의 차이가 곧 **척도 정렬의 실무적 대가**다")

run.finish({
    "exp_id": "quest46_q4e_operating_point",
    "metric": "global_threshold_rate_deviation_TE_minus_Aem",
    "value": float(H5["dev"]["mean"]),
    "passed": bool(ok_("H0") and ok_("H2") and ok_("H5")),
    "summary": ("동작점을 처음으로 쟀고(민감도·PPV·환자당 오경보) 축을 결정했다. "
                "H1 이 눈금을 세웠다 — PR-AUC 의 무작위 기저선은 0.5 가 아니라 유병률이고, "
                "Q2/Q7-B′ 의 매크로 0.8842 는 AUROC 라 Q4 라인의 PR-AUC 와 비교할 수 없다"
                "(같은 코호트에서 둘 다 냈다). H2 가 축을 정했다 — 레코드별 상수 시프트는 "
                "레코드 내 지표를 정확히 불변으로 두므로 **층②로 매크로를 올리는 것은 "
                "원리적으로 불가능**하다. H3 이 층②에 남은 천장을 쟀고, H4/H5 가 환자별 "
                "예산과 전역 단일 문턱에서 위양성·위음성을 직접 셌다. Q4-D 의 미검사 결함"
                "(Platt 기울기)도 H0 에서 막았다."),
    "verdicts": VERD, "notes": NOTE, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "H0": CONFIG.get("H0", {}),
    "H1": CONFIG.get("H1", {}), "H2": CONFIG.get("H2", {}), "H3": CONFIG.get("H3", {}),
    "H5": CONFIG.get("H5", {}), "need": CONFIG.get("need", {}),
    "H6": CONFIG.get("H6", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4e_operating_point.ipynb`")
